In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import torchvision.transforms as transforms


In [2]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])


In [3]:
trainset=CIFAR10(root="./data",train=True,download=True,transform=transform)
testset=CIFAR10(root="./data",train=False,download=True,transform=transform)


Files already downloaded and verified
Files already downloaded and verified


In [4]:
#Dataset Loader
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)

In [11]:
#Now The next Thing is to build our Model 
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),#Here Kernel is 2 ans stride is also 2

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),#Here Kernel is 2 ans stride is also 2

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),#Here Kernel is 2 ans stride is also 2
        )

        self.fc_layer=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
            
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layer(x)
        return x

In [12]:
model=CNN()
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [20]:
#Now Basically training the Model
epochs=5
for epoch in range(epochs):
   epoch_training_loss=0.0
   for images,lables in trainloader:
        optimizer.zero_grad()
        outputs=model.forward(images)
        loss=criterion(outputs,lables)
        loss.backward()
        optimizer.step()
        epoch_training_loss+=loss.item()
   print(f"for epoch the training loss is : {epoch+1}/{epochs} the training loss is {epoch_training_loss/len(trainloader)}")


for epoch the training loss is : 1/5 the training loss is 0.08612544597853022
for epoch the training loss is : 2/5 the training loss is 0.07553746345776903
for epoch the training loss is : 3/5 the training loss is 0.07625161482067659
for epoch the training loss is : 4/5 the training loss is 0.06790916979714247
for epoch the training loss is : 5/5 the training loss is 0.06878273535693717


In [19]:
#The Next Step is to evaluate the Model
correct=0
total=0
with torch.no_grad():
    for images,lables in testloader:
        outputs=model.forward(images)
        _,predicted=torch.max(outputs,1)
        correct+=(predicted==lables).sum().item()
        total+=lables.size(0)
    print(f"The accuracy  for the Model is : {correct/total}")

The accuracy  for the Model is : 0.7467
